<a href="https://colab.research.google.com/github/ramithuh/differentiable-trees/blob/main/Sankoff_Algorithm_in_JAX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import jax
import numpy as np
import jax.nn as nn
import jax.numpy as jnp
import matplotlib.pyplot as plt


### Define adjacency matrix

In [2]:
adj = jnp.array([[0,0,0,0,0,1,0,0,0],
                 [0,0,0,0,0,1,0,0,0],
                 [0,0,0,0,0,0,0,1,0],
                 [0,0,0,0,0,0,1,0,0],
                 [0,0,0,0,0,0,1,0,0],
                 [0,0,0,0,0,0,0,0,1],
                 [0,0,0,0,0,0,0,1,0],
                 [0,0,0,0,0,0,0,0,1],
                 [0,0,0,0,0,0,0,0,0]])

### Define cost matrix and leaf sequences

In [3]:
cost_mat = jnp.array([[0.  ,2.5 ,1   ,2.5],
                      [2.5 ,0.  ,2.5 ,1. ],
                      [1.  ,2.5 ,0   ,2.5],
                      [2.5 ,1.  ,2.5 ,0  ]])

seq = jnp.array([[1],
                 [0],
                 [1],
                 [0],
                 [2]])

In [4]:
n_nodes = adj.shape[0]
n_letters = 4
n_leaves = 5




In [5]:
def init_dp(dp):
  for i in range(0,n_leaves):
    dp =  dp.at[i,seq[i]].set(0)

  return dp

In [6]:
def run_dp(dp, dp_nodes, verbose = False):
  dp = init_dp(dp)

  for node in range(n_leaves, n_nodes):
    ind = jnp.argpartition(adj[:,node], -1)[-2:]

    left_node  = ind[0]
    right_node = ind[1]

    if(verbose):
      print(f"at node {node+1} children are : {left_node+1} and {right_node+1}")

    for i in range(n_letters):
    
      left_char = None
      left_cost = 1e3
      right_char = None
      right_cost = 1e3
  
      for l_c in range(0,n_letters):  ## left
        cost = cost_mat[i][l_c] + dp[left_node][l_c]
        if(cost < left_cost):
          left_cost = cost
          left_char = l_c

      for r_c in range(0,n_letters):  ## right
        cost = cost_mat[i][r_c] + dp[right_node][r_c]
        if(cost < right_cost):
          right_cost = cost
          right_char = r_c

      dp       = dp.at[node,i].set(left_cost + right_cost)
      dp_nodes[node][i] = {str(left_node) : left_char, str(right_node) : right_char}

  return dp, dp_nodes

In [7]:
dp_nodes = [[0 for i in range(n_letters)] for j in range(n_nodes)]

dp = jnp.ones((n_nodes, n_letters))*1e3
dp, connections = run_dp(dp,dp_nodes)

In [8]:
dp

Array([[1000. ,    0. , 1000. , 1000. ],
       [   0. , 1000. , 1000. , 1000. ],
       [1000. ,    0. , 1000. , 1000. ],
       [   0. , 1000. , 1000. , 1000. ],
       [1000. , 1000. ,    0. , 1000. ],
       [   2.5,    2.5,    3.5,    3.5],
       [   1. ,    5. ,    1. ,    5. ],
       [   3.5,    3.5,    3.5,    4.5],
       [   6. ,    6. ,    7. ,    8. ]], dtype=float32)

In [9]:
connections

[[0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [{'0': 1, '1': 0}, {'0': 1, '1': 0}, {'0': 1, '1': 0}, {'0': 1, '1': 0}],
 [{'3': 0, '4': 2}, {'3': 0, '4': 2}, {'3': 0, '4': 2}, {'3': 0, '4': 2}],
 [{'2': 1, '6': 0}, {'2': 1, '6': 0}, {'2': 1, '6': 2}, {'2': 1, '6': 0}],
 [{'5': 0, '7': 0}, {'5': 1, '7': 1}, {'5': 0, '7': 2}, {'5': 1, '7': 1}]]

### Backtrack and find the actual characters

In [10]:
def backtrack_and_find_nodes(node, letter, dp, connections):
  global seq_chars

  print(node)

  if(node <= n_letters):
    return 

  seq_chars = seq_chars.at[node].set(letter)
  # print(seq_chars)

  children_info = connections[node][letter]
  child = list(children_info.keys())

  # print(int(child[0]), children_info[int(child[0])])
  backtrack_and_find_nodes(int(child[0]), children_info[child[0]], dp, connections)
  backtrack_and_find_nodes(int(child[1]), children_info[child[1]], dp, connections)





In [11]:
node = adj.shape[0] - 1
letter = jnp.argmin(dp[node,:])

In [12]:
seq_chars = jnp.zeros((n_nodes,1)) 

backtrack_and_find_nodes(node,0, dp, connections)

8
5
0
1
7
2
6
3
4


In [13]:
seq_chars

Array([[0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.]], dtype=float32)